In [5]:
!pip install xgboost==3.4.1

In [7]:
import joblib

model_bundle = joblib.load(
    "models/churn_model_bundle.joblib"
)

final_xgb_model = model_bundle["model"]
selected_threshold = model_bundle["threshold"]

print("Model loaded successfully!")
print("Selected threshold:", selected_threshold)

In [9]:
print("Bundle keys:", model_bundle.keys())

print("\nModel type:")
print(type(model_bundle["model"]))

print("\nPipeline steps:")
print(model_bundle["model"].steps)

print("\nThreshold:")
print(model_bundle["threshold"])

print("\nModel file size:")
import os
print(os.path.getsize("models/churn_model_bundle.joblib"))

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Load cleaned dataset
df = pd.read_csv("data/processed/cleaned_clients.csv")

# Recreate feature engineering
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["ServiceCount"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0,
        "No phone service": 0
    })
    .sum(axis=1)
)

# Separate features and target
X = df.drop(columns=["Churn", "customerID"])

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Recreate the exact original split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dataset restored!")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [4]:
import pandas as pd
import numpy as np

# Define risk classification function
def classify_customer_risk(probability):

    if probability < 0.20:
        return "LOW"

    elif probability < 0.40:
        return "MEDIUM"

    else:
        return "HIGH"

In [5]:
test_probabilities = [0.05, 0.25, 0.39, 0.65, 0.92]

for probability in test_probabilities:

    risk = classify_customer_risk(probability)

    print(
        f"Probability: {probability:.0%} | Risk: {risk}"
    )

In [6]:
# Generate churn probabilities
test_probabilities = final_xgb_model.predict_proba(
    X_test
)[:, 1]

# Generate binary predictions
test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

# Classify risk
def classify_customer_risk(probability):

    if probability < 0.20:
        return "LOW"

    elif probability < 0.40:
        return "MEDIUM"

    else:
        return "HIGH"


# Build customer risk dataframe
customer_risk_df = pd.DataFrame({
    "Churn_Probability": test_probabilities,
    "Predicted_Churn": test_predictions,
    "Risk_Category": [
        classify_customer_risk(p)
        for p in test_probabilities
    ],
    "Actual_Churn": y_test.values
}, index=X_test.index)

# Add customer IDs
customer_risk_df.insert(
    0,
    "CustomerID",
    df.loc[X_test.index, "customerID"]
)

display(customer_risk_df.head(10))

In [7]:
import os

os.makedirs("../data/processed", exist_ok=True)

customer_risk_df.to_csv(
    "../data/processed/customer_risk_predictions.csv",
    index=True,
    index_label="original_index"
)

print("Customer risk predictions saved!")
print("Total records:", len(customer_risk_df))

In [8]:
risk_distribution = (
    customer_risk_df["Risk_Category"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"], fill_value=0)
)

risk_summary = pd.DataFrame({
    "Customer_Count": risk_distribution,
    "Percentage": (
        risk_distribution / len(customer_risk_df) * 100
    ).round(2)
})

display(risk_summary)

In [9]:
# Compare predicted risk categories with actual churn
risk_validation = pd.crosstab(
    customer_risk_df["Risk_Category"],
    customer_risk_df["Actual_Churn"],
    margins=True
)

display(risk_validation)

In [10]:
risk_performance = (
    customer_risk_df
    .groupby("Risk_Category")
    .agg(
        Customer_Count=("Actual_Churn", "count"),
        Actual_Churners=("Actual_Churn", "sum"),
        Actual_Churn_Rate=("Actual_Churn", "mean")
    )
)

risk_performance["Actual_Churn_Rate"] *= 100

risk_performance = risk_performance.reindex(
    ["LOW", "MEDIUM", "HIGH"]
)

display(risk_performance.round(2))

In [11]:
monthly_charge_limit = X_train["MonthlyCharges"].quantile(0.75)

print(
    "Monthly charge benchmark:",
    round(monthly_charge_limit, 2)
)

In [12]:
import pandas as pd

def generate_retention_recommendations(customer):

    recommendations = []

    # Contract-related recommendation
    if customer["Contract"] == "Month-to-month":
        recommendations.append({
            "Reason": "Month-to-month contract",
            "Action": (
                "Present suitable annual or two-year "
                "contract options and explain their benefits."
            ),
            "Priority": "High"
        })

    # Short tenure
    if customer["tenure"] <= 12:
        recommendations.append({
            "Reason": "Short customer tenure",
            "Action": (
                "Provide onboarding assistance and "
                "schedule a customer satisfaction check-in."
            ),
            "Priority": "High"
        })

    # Online security
    if customer["OnlineSecurity"] == "No":
        recommendations.append({
            "Reason": "No online security service",
            "Action": (
                "Explain available online security services "
                "and offer a consultation."
            ),
            "Priority": "Medium"
        })

    # Technical support
    if customer["TechSupport"] == "No":
        recommendations.append({
            "Reason": "No technical support service",
            "Action": (
                "Explain available technical support "
                "plans and assistance options."
            ),
            "Priority": "Medium"
        })

    # Higher monthly charges
    if customer["MonthlyCharges"] >= monthly_charge_limit:
        recommendations.append({
            "Reason": "Higher monthly charges",
            "Action": (
                "Review the customer's current plan and "
                "identify suitable cost-effective alternatives."
            ),
            "Priority": "Medium"
        })

    # Electronic check payment
    if customer["PaymentMethod"] == "Electronic check":
        recommendations.append({
            "Reason": "Electronic check payment method",
            "Action": (
                "Review the payment experience and explain "
                "available payment options."
            ),
            "Priority": "Low"
        })

    if not recommendations:
        recommendations.append({
            "Reason": "No specific rule-based concern identified",
            "Action": (
                "Continue regular customer engagement "
                "and monitor churn risk."
            ),
            "Priority": "Low"
        })

    return recommendations

In [13]:
recommendation_records = []

for idx, row in X_test.iterrows():

    risk = customer_risk_df.loc[idx, "Risk_Category"]

    recommendations = generate_retention_recommendations(row)

    recommendation_records.append({
        "CustomerID": df.loc[idx, "customerID"],
        "Churn_Probability": customer_risk_df.loc[
            idx, "Churn_Probability"
        ],
        "Risk_Category": risk,
        "Recommendations": recommendations
    })

recommendation_df = pd.DataFrame(recommendation_records)

display(recommendation_df.head(10))

In [14]:
high_risk_customers = recommendation_df[
    recommendation_df["Risk_Category"] == "HIGH"
]

sample_customer = high_risk_customers.iloc[0]

print("Customer ID:", sample_customer["CustomerID"])

print(
    "Churn Probability:",
    round(sample_customer["Churn_Probability"] * 100, 2),
    "%"
)

print("Risk:", sample_customer["Risk_Category"])

print("\nRecommended Actions:")

for item in sample_customer["Recommendations"]:

    print("\nReason:", item["Reason"])
    print("Action:", item["Action"])
    print("Priority:", item["Priority"])

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2cb50308-81e6-4757-a2d6-155b4c4296dc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>